# Lab 3 – Security Testing & Red Teaming

**Estimated Time:** 110–150 minutes  
**Difficulty:** Advanced

## Learning Objectives
- Conduct structured threat modeling for enterprise LLM applications
- Build automated prompt-injection and jailbreak test harnesses
- Orchestrate red team attack suites with coverage reporting
- Implement response classification, triage, and incident logging
- Generate executive-ready security summaries and mitigation plans

## Prerequisites & Setup
- Python 3.10+ environment with `httpx`, `pandas`, `numpy`, `rich`, `jinja2`, `langfuse`, and `openai` installed
- Access to an evaluation sandbox endpoint (real or mocked) exposed via REST; configure `EVAL_ENDPOINT` env var
- Optional: Vulnerable demo LLM endpoint to execute live red team attacks
- API keys for evaluation models stored in environment variables (`OPENAI_API_KEY`, etc.)
- Sample dataset of historical incidents (`data/incidents.csv`) for analytics (provide stub if unavailable)

> **Tip:** If you lack a live endpoint, stub `send_prompt` to return canned responses so you can complete the exercises.

## Lab Structure
You will complete eight exercises focused on systematically probing and hardening LLM-enabled systems.

1. Threat modeling and risk catalog
2. Prompt injection detection harness
3. Attack generation templates
4. Response evaluation and policy classification
5. Red team orchestrator with coverage metrics
6. Telemetry, logging, and forensic capture
7. Incident severity scoring and triage workflows
8. Executive reporting and mitigation planning

Each exercise contains starter code with `TODO` markers. Complete the logic, run the cell, and capture findings in the provided reflection prompts.

## Exercise 1: Build a Threat Model Catalog
**Scenario:** Security leadership needs a living catalog of threats mapped to attack surfaces and mitigations.

**Success Criteria**
- Threat actors, assets, and mitigations represented using dataclasses
- Catalog supports registering new threats and exporting summaries
- Risk scoring combines likelihood, impact, and detection difficulty

**Hints**
- Use enums for threat categories (prompt injection, data exfiltration, model theft, etc.)
- Provide helper to compute composite risk (e.g., weighted average)
- Include method to render catalog as Markdown or JSON for reporting

In [ ]:
from __future__ import annotations
import enum
from dataclasses import dataclass, field
from typing import Dict, List, Optional

class ThreatCategory(enum.Enum):
    PROMPT_INJECTION = "prompt_injection"
    DATA_EXFILTRATION = "data_exfiltration"
    MODEL_THEFT = "model_theft"
    SAFETY_BYPASS = "safety_bypass"
    SUPPLY_CHAIN = "supply_chain"

@dataclass
class ThreatEntry:
    name: str
    category: ThreatCategory
    attack_surface: str
    likelihood: int
    impact: int
    detection_difficulty: int
    mitigations: List[str] = field(default_factory=list)
    notes: Optional[str] = None

    def risk_score(self) -> float:
        """Compute composite risk score (0-10)."""
        # TODO: Combine likelihood, impact, and detection difficulty into a weighted score
        ...

class ThreatCatalog:
    def __init__(self) -> None:
        self._entries: Dict[str, ThreatEntry] = {}

    def register(self, entry: ThreatEntry) -> None:
        # TODO: Insert entry with defensive duplicate handling and logging hook
        ...

    def to_markdown(self) -> str:
        # TODO: Render entries as Markdown table sorted by risk_score
        ...

    def export(self, fmt: str = "json") -> str:
        # TODO: Support JSON (default) and Markdown exports
        ...

# TODO: Instantiate catalog, register sample threats, and preview Markdown export

**Reflection & Verification**
- [ ] Catalog captures at least three threat categories with mitigations
- [ ] Risk scores align with qualitative expectations
- [ ] Export format ready for sharing with security stakeholders

## Exercise 2: Prompt Injection Detection Harness
**Scenario:** You need an automated harness that probes prompts for injection patterns before sending them to the model.

**Success Criteria**
- Maintains heuristics (regex, keyword lists) for suspicious constructs
- Supports pluggable detectors, including user-supplied callables
- Provides structured result with severity, triggers, and remediation advice

**Hints**
- Combine fast heuristics with optional ML classifier hooks
- Allow detectors to short-circuit on critical findings
- Track detection latency for performance monitoring

In [ ]:
import re
import time
from typing import Any, Callable, List, NamedTuple

Suspicion = NamedTuple("Suspicion", [
    ("severity", str),
    ("reason", str),
    ("detector", str),
])

class InjectionHarness:
    def __init__(self) -> None:
        self.regex_rules: List[re.Pattern[str]] = []
        self.callbacks: List[Callable[[str], Optional[Suspicion]]] = []

    def register_regex(self, pattern: str) -> None:
        # TODO: Compile pattern defensively and append to regex_rules
        ...

    def register_callback(self, detector: Callable[[str], Optional[Suspicion]]) -> None:
        # TODO: Append detector callable with type/exception safety
        ...

    def analyze(self, prompt: str) -> Dict[str, Any]:
        started = time.perf_counter()
        findings: List[Suspicion] = []
        # TODO: Apply regex_rules and callbacks, capturing first critical hit
        # TODO: Return dict with findings, max severity, latency, and recommended action
        ...

# TODO: Configure harness with baseline heuristics and test against benign/malicious prompts

**Reflection & Verification**
- [ ] Harness flags classic injection attempts (ignore previous, do X)
- [ ] Latency remains <10ms for heuristic-only paths
- [ ] Result payload consumable by upstream guardrail pipeline

## Exercise 3: Attack Generation Templates
**Scenario:** Scale red teaming by generating families of adversarial prompts programmatically.

**Success Criteria**
- Template system expands attack archetypes (jailbreak, extraction, policy bypass)
- Supports parameter grids (tones, obfuscation levels, languages)
- Produces metadata describing each generated attack

**Hints**
- Use `itertools.product` for combinatorial generation
- Consider YAML/JSON config for loading template definitions
- Track provenance (template id, parameters) for reproducibility

In [ ]:
import itertools
from pathlib import Path
import json

def load_templates(path: Path) -> Dict[str, Dict[str, Any]]:
    # TODO: Load JSON/YAML template definitions containing prompt skeletons
    ...

def generate_attacks(templates: Dict[str, Dict[str, Any]]) -> List[Dict[str, Any]]:
    attacks: List[Dict[str, Any]] = []
    # TODO: Iterate across templates, expand parameter grids, and build attack payloads
    ...
    return attacks

# TODO: Provide inline fallback templates if file not present and preview first five attacks

**Reflection & Verification**
- [ ] Generated attacks cover multiple categories and languages
- [ ] Metadata sufficient to replay specific attack cases
- [ ] Template loader gracefully handles missing/invalid config

## Exercise 4: Response Evaluation & Policy Classification
**Scenario:** Automatically classify model responses to determine policy compliance.

**Success Criteria**
- Evaluation function inspects responses for policy breaches (PII leak, disallowed content)
- Severity grades mapped to response handling (`allow`, `warn`, `block`)
- Supports pluggable evaluation strategies (regex, embedding similarity, external moderation APIs)

**Hints**
- Use dataclasses or NamedTuple to structure evaluation results
- Pipeline should capture evidence snippets for auditor review
- Provide summary stats (counts per severity)

In [ ]:
from dataclasses import dataclass

@dataclass
class EvaluationResult:
    severity: str
    reasons: List[str]
    evidence: List[str]
    action: str

class ResponseEvaluator:
    def __init__(self) -> None:
        self.checks: List[Callable[[str], Optional[EvaluationResult]]] = []

    def register_check(self, check: Callable[[str], Optional[EvaluationResult]]) -> None:
        # TODO: Store check with duplicate prevention and exception safety
        ...

    def evaluate(self, text: str) -> EvaluationResult:
        # TODO: Run registered checks, aggregate findings, and compute final severity/action
        ...

    def summary(self) -> Dict[str, int]:
        # TODO: Maintain running counts of severities for reporting
        ...

# TODO: Implement baseline checks (PII regex, disallowed keywords) and test evaluator

**Reflection & Verification**
- [ ] Classifier differentiates allow/warn/block cases
- [ ] Evidence traces recorded for each violation
- [ ] Summary metrics align with manual spot-checks

## Exercise 5: Red Team Orchestrator & Coverage Metrics
**Scenario:** Combine attack generation, harness analysis, and evaluation into a repeatable red team run.

**Success Criteria**
- Orchestrator coordinates sending attacks, capturing responses, and storing results
- Computes coverage metrics (attack families attempted vs. total)
- Supports retry/backoff and configurable concurrency

**Hints**
- Use asyncio with semaphore to bound concurrency
- Persist run artifacts to disk (JSONL) for reproducibility
- Provide hooks to integrate with Langfuse for trace ingestion

In [ ]:
import asyncio
import os
import json
from pathlib import Path

async def send_prompt(prompt: str) -> str:
    # TODO: Implement HTTP call to evaluation endpoint or stubbed response
    ...

class RedTeamRun:
    def __init__(self, attacks: List[Dict[str, Any]], harness: InjectionHarness, evaluator: ResponseEvaluator) -> None:
        self.attacks = attacks
        self.harness = harness
        self.evaluator = evaluator
        self.results: List[Dict[str, Any]] = []

    async def execute(self, concurrency: int = 5) -> None:
        semaphore = asyncio.Semaphore(concurrency)

        async def worker(attack: Dict[str, Any]) -> None:
            async with semaphore:
                # TODO: Run harness, short-circuit if blocked, otherwise send prompt and evaluate response
                ...

        await asyncio.gather(*(worker(attack) for attack in self.attacks))

    def coverage_report(self) -> Dict[str, Any]:
        # TODO: Summarize attempts and success rates per attack family
        ...

    def persist(self, path: Path) -> None:
        # TODO: Write results to disk as JSONL with metadata header
        ...

# TODO: Demonstrate orchestrator with sample attacks and print coverage metrics

**Reflection & Verification**
- [ ] Orchestrator handles at least 50 attacks without errors
- [ ] Coverage report highlights uncovered attack families
- [ ] Results persisted for audit trail

## Exercise 6: Telemetry, Logging & Forensics
**Scenario:** Capture detailed telemetry for each attack to support incident investigations.

**Success Criteria**
- Structured logging includes correlation IDs, timestamps, and artifact links
- Optional integration with Langfuse traces for cross-team visibility
- Provides utility to snapshot offending prompts/responses to secure storage

**Hints**
- Use `uuid.uuid4()` for correlation IDs
- Redact secrets before storing artifacts
- Consider asynchronous file I/O if logging volume is high

In [ ]:
import uuid
from datetime import datetime

def record_event(store: List[Dict[str, Any]], *, stage: str, prompt: str, response: Optional[str], metadata: Dict[str, Any]) -> str:
    # TODO: Generate correlation ID, redact sensitive fields, append to store
    ...

def snapshot_artifacts(correlation_id: str, prompt: str, response: str, directory: Path) -> Path:
    # TODO: Persist prompt/response pair to secure directory with redaction
    ...

# TODO: Hook logging functions into RedTeamRun workflow and validate artifact outputs

**Reflection & Verification**
- [ ] Every red team event has a correlation ID and timestamp
- [ ] Artifacts stored without leaking sensitive data
- [ ] Telemetry integrates with existing observability stack

## Exercise 7: Incident Severity Scoring & Triage
**Scenario:** Translate evaluation signals into incident tickets with consistent severity scoring.

**Success Criteria**
- Severity score accounts for policy breach type, exploitability, and business context
- Outputs triage recommendation (immediate action, monitor, backlog)
- Integrates with existing incident tracker via stubbed API client

**Hints**
- Use configurable weights for scoring dimensions
- Provide serializer to produce ticket payload (JSON)
- Support suppression rules for known acceptable failures

In [ ]:
def compute_severity(result: EvaluationResult, context: Dict[str, Any]) -> Dict[str, Any]:
    # TODO: Map evaluation severity + context (business unit, data class) to numeric score
    ...

def submit_incident(ticket: Dict[str, Any]) -> None:
    # TODO: Integrate with incident tracker API or simulate submission
    ...

# TODO: Create sample evaluation results, compute severity, and generate ticket preview

**Reflection & Verification**
- [ ] Severity scores prioritize high-impact violations
- [ ] Tickets contain actionable remediation guidance
- [ ] Suppression rules documented and justified

## Exercise 8: Executive Reporting & Mitigation Plan
**Scenario:** Summarize red team outcomes for executives with clear mitigation roadmap.

**Success Criteria**
- Aggregates red team run data into high-level metrics (pass/fail rate, top risks)
- Generates report using Jinja2 template (Markdown/HTML)
- Includes prioritized mitigation plan with owners and timelines

**Hints**
- Reuse coverage metrics and incident severities from earlier exercises
- Provide both machine-readable (JSON) and human-readable outputs
- Validate report renders even when some data sources are missing

In [ ]:
from jinja2 import Template

REPORT_TEMPLATE = ""
# TODO: Provide Jinja2 template string with placeholders for metrics, risks, mitigations
"""

def build_report(run_summary: Dict[str, Any], incidents: List[Dict[str, Any]]) -> Dict[str, str]:
    # TODO: Compute aggregate metrics and render template to Markdown/HTML
    ...

def plan_mitigations(incidents: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # TODO: Prioritize incidents and assign owners/timelines
    ...

# TODO: Mock run summary + incidents, render report preview, and print mitigation plan

**Reflection & Verification**
- [ ] Report communicates key metrics and next steps clearly
- [ ] Missing data handled gracefully (placeholders or warnings)
- [ ] Mitigation plan aligns with severity prioritization

## Wrap-Up
You built a comprehensive security testing workflow covering threat modeling, automated attack generation, response evaluation, orchestration, telemetry, and executive reporting. Continue by:
- Integrating red team runs into CI/CD or nightly regression suites
- Stress-testing guardrails with live adversarial research partners
- Feeding incident learnings back into guardrail and training datasets
- Collaborating with compliance to align playbooks with regulatory obligations

## Submission Checklist
- [ ] All `TODO` blocks implemented and validated
- [ ] Red team orchestrator run completed with coverage report attached
- [ ] Incident tickets drafted for critical findings
- [ ] Executive report generated with mitigation plan